# Quickstart

This offline tutorial creates a spectrum with Astropy units, changes its resolving power and sampling, calculates a 2MASS J-band flux density, and plots the result. The synthetic absorption lines are teaching data, not a stellar atmosphere model.

In [ ]:
import astropy.units as u
import matplotlib.pyplot as plt
import numpy as np
import speclib

from speclib import Filter, Spectrum, apply_filter

print(f"speclib {speclib.__version__}")

## Create and inspect a spectrum

`Spectrum` extends `specutils.Spectrum1D`. Wavelength and flux remain Astropy quantities, so units are visible at every step.

In [ ]:
wavelength = np.linspace(10_000, 14_000, 8001) * u.AA
continuum = np.ones(wavelength.size)
line_1 = 0.25 * np.exp(-0.5 * ((wavelength.value - 11_500) / 2.0) ** 2)
line_2 = 0.15 * np.exp(-0.5 * ((wavelength.value - 12_800) / 3.0) ** 2)
flux_unit = u.erg / (u.s * u.cm**2 * u.AA)
spectrum = Spectrum(
    spectral_axis=wavelength,
    flux=(continuum - line_1 - line_2) * flux_unit,
)
print(spectrum.wavelength[[0, -1]])
print(spectrum.flux.unit, spectrum.flux.shape)

## Broaden, then resample

Broadening to constant $R=1000$ retains the exact input wavelength samples. Resampling is a separate operation. Both methods return new objects, leaving `spectrum` unchanged.

In [ ]:
broadened = spectrum.set_spectral_resolving_power(1000)
output_wavelength = np.linspace(10_100, 13_900, 381) * u.AA
sampled = broadened.resample(output_wavelength)

print(len(spectrum.wavelength), len(broadened.wavelength), len(sampled.wavelength))
print(np.array_equal(spectrum.wavelength, broadened.wavelength))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
region = (wavelength.value > 11_450) & (wavelength.value < 11_550)
ax.plot(spectrum.wavelength[region], spectrum.flux[region], label="input")
ax.plot(broadened.wavelength[region], broadened.flux[region], label="R = 1000")
ax.set(xlabel="Wavelength [Angstrom]", ylabel=r"Flux density [erg s$^{-1}$ cm$^{-2}$ Angstrom$^{-1}$]")
ax.legend();

## Apply a packaged filter

`apply_filter` resamples the dimensionless response to the spectrum axis if necessary, integrates flux times response, and divides by the tabulated bandwidth.

In [ ]:
j_band = Filter("2MASS J")
j_flux = apply_filter(spectrum, j_band)
print(f"2MASS J mean flux density: {j_flux:.4g}")

## Next steps

Use the tutorial on spectral grids for repeated parameter retrieval, and read the model library reference before downloading PHOENIX, SPHINX, or NewEra data. Real model loaders return Å and `erg / (s cm2 Å)` after converting the units used by the source product; consult the product page for its wavelength convention.